In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers torch scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import os
import json
from datetime import datetime

In [ ]:
# Configuration
CONFIG = {
    "model_name":    "distilbert-base-uncased",
    "max_length":    512,
    "batch_size":    16,
    "epochs":        3,
    "learning_rate": 2e-5,
    "warmup_steps":  100,
    "train_ratio":   0.70,
    "val_ratio":     0.15,
    "test_ratio":    0.15,
    "random_seed":   42,
    "checkpoint_dir": "/content/drive/MyDrive/cm3070_checkpoints",
    "csv_path":      "/content/sample_data/phishing_email.csv",
}

# Create checkpoint directory
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Config: {json.dumps(CONFIG, indent=2)}")

Using device: cuda
Config: {
  "model_name": "distilbert-base-uncased",
  "max_length": 512,
  "batch_size": 16,
  "epochs": 3,
  "learning_rate": 2e-05,
  "warmup_steps": 100,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  "random_seed": 42,
  "checkpoint_dir": "/content/drive/MyDrive/cm3070_checkpoints",
  "csv_path": "/content/sample_data/phishing_email.csv"
}


In [ ]:

print("Loading dataset...")
df = pd.read_csv(CONFIG["csv_path"])


print(f"Total rows:     {len(df):,}")
print(f"Columns:        {list(df.columns)}")
print(f"Class balance:")
print(df['label'].value_counts())
print(f"Nulls:          {df.isnull().sum().sum()}")

# Drop nulls
df = df.dropna(subset=['text_combined', 'label'])
df['label'] = df['label'].astype(int)

# Split
texts  = df['text_combined'].tolist()
labels = df['label'].tolist()

# First split: train vs (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=(1 - CONFIG["train_ratio"]),
    stratify=labels,
    random_state=CONFIG["random_seed"]
)

# Second split: val vs test
val_size = CONFIG["val_ratio"] / (1 - CONFIG["train_ratio"])
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=(1 - val_size),
    stratify=y_temp,
    random_state=CONFIG["random_seed"]
)

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train):,}")
print(f"  Val:   {len(X_val):,}")
print(f"  Test:  {len(X_test):,}")

Loading dataset...
Total rows:     82,486
Columns:        ['text_combined', 'label']
Class balance:
label
1    42891
0    39595
Name: count, dtype: int64
Nulls:          0

Split sizes:
  Train: 57,740
  Val:   12,372
  Test:  12,374


In [ ]:
# Load tokeniser
print("Loading tokeniser...")
tokenizer = DistilBertTokenizerFast.from_pretrained(CONFIG["model_name"])
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

def tokenise_batch(texts, tokenizer, max_length):
    """Tokenise a list of texts and return encoding dict."""
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

print("Tokenising training set...")
train_encodings = tokenise_batch(X_train, tokenizer, CONFIG["max_length"])

print("Tokenising validation set...")
val_encodings = tokenise_batch(X_val, tokenizer, CONFIG["max_length"])

print("Tokenising test set...")
test_encodings = tokenise_batch(X_test, tokenizer, CONFIG["max_length"])

print("Tokenisation complete.")
print(f"Train input_ids shape: {train_encodings['input_ids'].shape}")

Loading tokeniser...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Vocabulary size: 30,522
Tokenising training set...
Tokenising validation set...
Tokenising test set...
Tokenisation complete.
Train input_ids shape: torch.Size([57740, 512])


In [ ]:
class PhishingEmailDataset(Dataset):
    """
    PyTorch Dataset wrapping tokenised email encodings and labels.
    The DataLoader uses this to feed batches to the model during training.
    """

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        # Called by DataLoader to know how many samples exist
        return len(self.labels)

    def __getitem__(self, idx):
        # Called by DataLoader to fetch one sample by index
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = PhishingEmailDataset(train_encodings, y_train)
val_dataset   = PhishingEmailDataset(val_encodings,   y_val)
test_dataset  = PhishingEmailDataset(test_encodings,  y_test)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")
print(f"Test batches:  {len(test_loader):,}")

Train batches: 3,609
Val batches:   774
Test batches:  774


In [ ]:
print("Loading pre-trained DistilBERT...")
model = DistilBertForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2
)
model = model.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model on device:      {DEVICE}")

Loading pre-trained DistilBERT...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     66,955,010
Trainable parameters: 66,955,010
Model on device:      cuda


In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=0.01
)

total_steps = len(train_loader) * CONFIG["epochs"]

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG["warmup_steps"],
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps:         {CONFIG['warmup_steps']}")
print(f"Learning rate:        {CONFIG['learning_rate']}")

Total training steps: 10,827
Warmup steps:         100
Learning rate:        2e-05


In [ ]:
def evaluate(model, loader, device):
    """Run model on a dataloader, return loss and predictions."""
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return avg_loss, accuracy, all_preds, all_labels


# Training loop
print("Starting training...\n")
training_log = []

for epoch in range(1, CONFIG["epochs"] + 1):
    print(f"{'='*50}")
    print(f"Epoch {epoch} / {CONFIG['epochs']}")
    print(f"{'='*50}")

    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        # Clear gradients from previous batch
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backward pass, compute gradients
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if (batch_idx + 1) % 200 == 0:
            print(f"  Batch {batch_idx+1:,}/{len(train_loader):,} | "
                  f"Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    print(f"\nRunning validation...")
    val_loss, val_acc, _, _ = evaluate(model, val_loader, DEVICE)

    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Acc:    {val_acc:.4f}")

    checkpoint_path = os.path.join(
        CONFIG["checkpoint_dir"], f"epoch_{epoch}"
    )
    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)

    epoch_log = {
        "epoch":          epoch,
        "train_loss":     round(avg_train_loss, 4),
        "val_loss":       round(val_loss, 4),
        "val_accuracy":   round(val_acc, 4),
        "timestamp":      datetime.now().isoformat()
    }
    training_log.append(epoch_log)
    with open(os.path.join(CONFIG["checkpoint_dir"], "training_log.json"), "w") as f:
        json.dump(training_log, f, indent=2)

    print(f"  Checkpoint saved → {checkpoint_path}")

print("\nTraining complete.")

Starting training...

Epoch 1 / 3
  Batch 200/3,609 | Loss: 0.3571
  Batch 400/3,609 | Loss: 0.0053
  Batch 600/3,609 | Loss: 0.0742
  Batch 800/3,609 | Loss: 0.0034
  Batch 1,000/3,609 | Loss: 0.0020
  Batch 1,200/3,609 | Loss: 0.3948
  Batch 1,400/3,609 | Loss: 0.1743
  Batch 1,600/3,609 | Loss: 0.0007
  Batch 1,800/3,609 | Loss: 0.3290
  Batch 2,000/3,609 | Loss: 0.0021
  Batch 2,200/3,609 | Loss: 0.0036
  Batch 2,400/3,609 | Loss: 0.0067
  Batch 2,600/3,609 | Loss: 0.2905
  Batch 2,800/3,609 | Loss: 0.0004
  Batch 3,000/3,609 | Loss: 0.0003
  Batch 3,200/3,609 | Loss: 0.0002
  Batch 3,400/3,609 | Loss: 0.0002
  Batch 3,600/3,609 | Loss: 0.0014

Running validation...

Epoch 1 Summary:
  Train Loss: 0.0768
  Val Loss:   0.0314
  Val Acc:    0.9917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint saved → /content/drive/MyDrive/cm3070_checkpoints/epoch_1
Epoch 2 / 3
  Batch 200/3,609 | Loss: 0.0001
  Batch 400/3,609 | Loss: 0.0002
  Batch 600/3,609 | Loss: 0.0003
  Batch 800/3,609 | Loss: 0.0296
  Batch 1,000/3,609 | Loss: 0.0001
  Batch 1,200/3,609 | Loss: 0.0001
  Batch 1,400/3,609 | Loss: 0.0001
  Batch 1,600/3,609 | Loss: 0.0002
  Batch 1,800/3,609 | Loss: 0.0003
  Batch 2,000/3,609 | Loss: 0.0006
  Batch 2,200/3,609 | Loss: 0.0001
  Batch 2,400/3,609 | Loss: 0.0001
  Batch 2,600/3,609 | Loss: 0.0002
  Batch 2,800/3,609 | Loss: 0.0001
  Batch 3,000/3,609 | Loss: 0.0001
  Batch 3,200/3,609 | Loss: 0.0002
  Batch 3,400/3,609 | Loss: 0.0001
  Batch 3,600/3,609 | Loss: 0.0001

Running validation...

Epoch 2 Summary:
  Train Loss: 0.0154
  Val Loss:   0.0389
  Val Acc:    0.9932


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint saved → /content/drive/MyDrive/cm3070_checkpoints/epoch_2
Epoch 3 / 3
  Batch 200/3,609 | Loss: 0.0001
  Batch 400/3,609 | Loss: 0.0000
  Batch 600/3,609 | Loss: 0.0000
  Batch 800/3,609 | Loss: 0.0000
  Batch 1,000/3,609 | Loss: 0.0000
  Batch 1,200/3,609 | Loss: 0.0000
  Batch 1,400/3,609 | Loss: 0.0000
  Batch 1,600/3,609 | Loss: 0.0000
  Batch 1,800/3,609 | Loss: 0.0000
  Batch 2,000/3,609 | Loss: 0.0000
  Batch 2,200/3,609 | Loss: 0.0000
  Batch 2,400/3,609 | Loss: 0.0000
  Batch 2,600/3,609 | Loss: 0.0000
  Batch 2,800/3,609 | Loss: 0.0000
  Batch 3,000/3,609 | Loss: 0.0000
  Batch 3,200/3,609 | Loss: 0.0000
  Batch 3,400/3,609 | Loss: 0.0000
  Batch 3,600/3,609 | Loss: 0.0000

Running validation...

Epoch 3 Summary:
  Train Loss: 0.0032
  Val Loss:   0.0333
  Val Acc:    0.9946


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint saved → /content/drive/MyDrive/cm3070_checkpoints/epoch_3

Training complete.


In [ ]:
print("Loading best checkpoint for final evaluation...")
best_checkpoint = os.path.join(CONFIG["checkpoint_dir"], "epoch_3")

model = DistilBertForSequenceClassification.from_pretrained(best_checkpoint)
model = model.to(DEVICE)

print("Running test set evaluation...")
test_loss, test_acc, test_preds, test_labels = evaluate(
    model, test_loader, DEVICE
)

print(f"\n{'='*50}")
print(f"FINAL TEST RESULTS")
print(f"{'='*50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(
    test_labels, test_preds,
    target_names=["Legitimate", "Phishing"]
))

print("Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

# Comparison with Keras baseline
print(f"\n{'='*50}")
print(f"COMPARISON: Keras Baseline vs DistilBERT")
print(f"{'='*50}")
print(f"{'Metric':<25} {'Keras TF-IDF':<20} {'DistilBERT'}")
print(f"{'-'*65}")
print(f"{'Test Accuracy':<25} {'98.51%':<20} {test_acc*100:.2f}%")
print(f"{'Model type':<25} {'Dense NN':<20} {'Transformer'}")
print(f"{'Input representation':<25} {'TF-IDF (sparse)':<20} {'Contextual embeddings'}")
print(f"{'Parameters':<25} {'~330k':<20} {'~67M'}")

Loading best checkpoint for final evaluation...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Running test set evaluation...

FINAL TEST RESULTS
Test Loss:     0.0228
Test Accuracy: 0.9960

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      1.00      5940
    Phishing       1.00      1.00      1.00      6434

    accuracy                           1.00     12374
   macro avg       1.00      1.00      1.00     12374
weighted avg       1.00      1.00      1.00     12374

Confusion Matrix:
[[5922   18]
 [  32 6402]]

COMPARISON: Keras Baseline vs DistilBERT
Metric                    Keras TF-IDF         DistilBERT
-----------------------------------------------------------------
Test Accuracy             98.51%               99.60%
Model type                Dense NN             Transformer
Input representation      TF-IDF (sparse)      Contextual embeddings
Parameters                ~330k                ~67M


In [ ]:
final_model_path = "/content/distilbert_phishing_final"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Final model saved to: {final_model_path}")
print(f"\nFiles saved:")
for f in os.listdir(final_model_path):
    size = os.path.getsize(os.path.join(final_model_path, f))
    print(f"  {f:<40} {size/1024/1024:.1f} MB")

print("\nThis model is ready to load into FastAPI.")
print("Use: DistilBertForSequenceClassification.from_pretrained(path)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/distilbert_phishing_final

Files saved:
  config.json                              0.0 MB
  tokenizer_config.json                    0.0 MB
  tokenizer.json                           0.7 MB
  model.safetensors                        255.4 MB

This model is ready to load into FastAPI.
Use: DistilBertForSequenceClassification.from_pretrained(path)
